## Bijdrage Raoul Buffels — DeepLearningTeam10

Deze Citi Bike-uitwerking is aangeleverd als de bijdrage van **Raoul Buffels**. Teamleden: Jorre Van Dyck, Milan Wouters, Raoul Buffels en Andrew Noeyens. De concrete menselijke taakverdeling moet het team zelf aanvullen. **AI-gebruik:** Codex hielp met implementatie, uitleg, tests, uitvoering en verpakking.

De codecellen zijn uitgevoerd in de oorspronkelijke lokale `Citybike_full`-omgeving op 9 oktober 2026. Deze export behoudt die echte uitvoer; alleen deze Markdown-cel is toegevoegd. Lokale paden, timings en fingerprints in de rapporten beschrijven de oorspronkelijke uitvoering. Start voor heruitvoering vanuit `RaoulBuffels` of de map `notebooks` met de bijbehorende Python 3.11-omgeving. AWS is een afzonderlijke latere taak.

# 02 — Definitieve voorbereiding zonder grafieken

Bijdragen: door team in te vullen. `run.py prepare` verwerkt de bronnen geheel met code. Cachevalidatie gebruikt bron-, code- en partitionfingerprints. Voor een herbouw verwijder je alleen de gegenereerde data volgens README; de ruwe bronnen blijven behouden.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
locations = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'Citybike_full']
ROOT = next((p for p in locations if (p / 'citibike').exists()), None)
if ROOT is None: raise RuntimeError('Start dit notebook vanuit Citybike_full of notebooks.')
sys.path.insert(0, str(ROOT))
from citibike.config import REPORTS, DATA, MODELS
from citibike.datasets import manifest
from citibike.plots import show
def report(name):
    return json.loads((REPORTS / name).read_text(encoding='utf-8'))
def table(name):
    display(pd.read_csv(REPORTS / name, dtype={'start_station_id':'string','end_station_id':'string','station_id':'string'}))


De voorbereiding is uitgevoerd vóór dit notebook. Het manifest legt de volledige reproducerbare uitvoering vast.

In [2]:
m=manifest()
display(pd.DataFrame(m['partitions'])[['area','period','raw_rows','retained_rows','removed_rows','removed_percent','processing_seconds']])

,area,period,raw_rows,retained_rows,removed_rows,removed_percent,processing_seconds
0,JC,2015-09,6668,6668,0,0.000000,1.035295
1,JC,2015-10,19264,19264,0,0.000000,0.254332
2,JC,2015-11,15113,15113,0,0.000000,0.211582
3,JC,2015-12,11838,11838,0,0.000000,0.171406
4,JC,2016-01,7479,7479,0,0.000000,0.121079
...,...,...,...,...,...,...,...
286,NYC,2026-04,3860371,3860371,0,0.000000,46.068834
287,NYC,2026-05,4694878,4695001,512,0.010906,68.345987
288,NYC,2026-06,5384468,5384702,0,0.000000,73.327140
289,NYC,2026-07,4993137,4993722,0,0.000000,72.192812


Voorspeltijdstip en horizon worden vóór feature engineering getoond.

In [3]:
display(pd.DataFrame(report('prediction_contracts.json')['contracts']))

,target,prediction_time,horizon,available,forbidden
0,hourly_demand,00:00 America/New_York op voorspeldag D,alle 24 lokale kloklabels op D (23/25 fysieke ...,kalender van D; gemeten tellingen strikt vóór D,"tellingen eerder op D, weer achteraf, eindstat..."
1,daily_demand,00:00 America/New_York op voorspeldag D,volledig dagtotaal op D,kalender van D; historie strikt vóór D,delen van dagtotaal D; informatie die pas op D...
2,duration,bij vertrek van één rit,duur tot beëindiging van deze rit (geen vaste ...,"starttijd, startstation, bekende gebruiker- en...","eindtijd, eindstation, ritduur; nog niet afger..."


Het dagvooruitcontract laat uitsluitend afgeronde dagen als featurebron toe. De doelkeuze ligt al vast op ontwikkelingsdata.

In [4]:
from citibike.features import make_features, FEATURES
history=pd.read_parquet(DATA/'NYC_hourly_development.parquet')
features=make_features(history)
display(features.tail(8))

,timestamp,rides,area,day,hour_sin,hour_cos,weekday_sin,weekday_cos,year_sin,year_cos,is_weekend,clock_hours,lag_1d,lag_2d,lag_7d,lag_14d,same_hour_mean_7d,previous_day_total
112864,2026-04-30 16:00:00,10239,NYC,2026-04-30,-0.866025,-5.000000e-01,0.433884,-0.900969,0.888682,-0.458524,0,1,10237.0,10576.0,12655.0,12516.0,9992.857143,141454.0
112865,2026-04-30 17:00:00,14418,NYC,2026-04-30,-0.965926,-2.588190e-01,0.433884,-0.900969,0.888682,-0.458524,0,1,15303.0,16254.0,18064.0,18579.0,13326.857143,141454.0
112866,2026-04-30 18:00:00,14267,NYC,2026-04-30,-1.000000,-1.836970e-16,0.433884,-0.900969,0.888682,-0.458524,0,1,13780.0,14750.0,17433.0,17956.0,12279.571429,141454.0
112867,2026-04-30 19:00:00,10827,NYC,2026-04-30,-0.965926,2.588190e-01,0.433884,-0.900969,0.888682,-0.458524,0,1,9591.0,10592.0,12671.0,13308.0,8863.857143,141454.0
112868,2026-04-30 20:00:00,7266,NYC,2026-04-30,-0.866025,5.000000e-01,0.433884,-0.900969,0.888682,-0.458524,0,1,5100.0,6826.0,8210.0,9441.0,5562.142857,141454.0
112869,2026-04-30 21:00:00,5418,NYC,2026-04-30,-0.707107,7.071068e-01,0.433884,-0.900969,0.888682,-0.458524,0,1,2645.0,5156.0,6557.0,7253.0,3941.714286,141454.0
112870,2026-04-30 22:00:00,4439,NYC,2026-04-30,-0.500000,8.660254e-01,0.433884,-0.900969,0.888682,-0.458524,0,1,1449.0,3939.0,5855.0,5865.0,3102.857143,141454.0
112871,2026-04-30 23:00:00,2875,NYC,2026-04-30,-0.258819,9.659258e-01,0.433884,-0.900969,0.888682,-0.458524,0,1,620.0,2571.0,3667.0,3831.0,2091.142857,141454.0


In [5]:
display(pd.DataFrame(report('leakage_audit.json')))
display(pd.DataFrame(report('chronological_splits.json')['folds']))

,feature,available_at,available_at_prediction,availability_assumption,target_information,leaks
0,hour_sin,kalender bekend vóór 00:00 op D,True,kalender deterministisch bekend,geen doelwaarden,False
1,hour_cos,kalender bekend vóór 00:00 op D,True,kalender deterministisch bekend,geen doelwaarden,False
2,weekday_sin,kalender bekend vóór 00:00 op D,True,kalender deterministisch bekend,geen doelwaarden,False
3,weekday_cos,kalender bekend vóór 00:00 op D,True,kalender deterministisch bekend,geen doelwaarden,False
4,year_sin,kalender bekend vóór 00:00 op D,True,kalender deterministisch bekend,geen doelwaarden,False
5,year_cos,kalender bekend vóór 00:00 op D,True,kalender deterministisch bekend,geen doelwaarden,False
6,is_weekend,kalender bekend vóór 00:00 op D,True,kalender deterministisch bekend,geen doelwaarden,False
7,clock_hours,kalender bekend vóór 00:00 op D,True,kalender deterministisch bekend,geen doelwaarden,False
8,lag_1d,uitsluitend volledige dagen vóór D; rolling ee...,voorwaardelijk: externe tellingen bekend vóór D,Offline backtest veronderstelt tijdige externe...,historische doelwaarden toegestaan,False
9,lag_2d,uitsluitend volledige dagen vóór D; rolling ee...,voorwaardelijk: externe tellingen bekend vóór D,Offline backtest veronderstelt tijdige externe...,historische doelwaarden toegestaan,False


,train_start,train_stop,validation_start,validation_stop,train_last_day,validation_first_day,validation_last_day
0,0,109512,109512,110184,2025-12-11,2025-12-12,2026-01-08
1,0,110184,110184,110856,2026-01-08,2026-01-09,2026-02-05
2,0,110856,110856,111528,2026-02-05,2026-02-06,2026-03-05
3,0,111528,111528,112200,2026-03-05,2026-03-06,2026-04-02
4,0,112200,112200,112872,2026-04-02,2026-04-03,2026-04-30
